In [14]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()

model = "claude-sonnet-4-5"


### Making my first request

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

print(message.content[0].text)


### Multi- Turn Conversations 
### Building helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# putting into practice 

messages = []

add_user_message(messages, "Define quantum computing in one sentence")

answer = chat(messages)

add_assistant_message(messages, answer)

add_user_message(messages, "Write Another Sentence")

final_answer = chat(messages)



### Lesson 5 System prompts
# building a flexible cha fucntion

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }


    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Now you can call the chat function with or without a system promt:

# Without system prompt
answer = chat(messages)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)

print(answer)




Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can solve certain complex problems much faster than classical computers.
Quantum computers use quantum bits (qubits) that can exist in multiple states simultaneously, unlike classical bits that are either 0 or 1, enabling them to explore many possible solutions at once.


### Prompting a eval workflow 

In [21]:
# This prompt will be the baseline for testing and improvement 

prompt1 = f"""
Please answer the user's question:

{"Whats 2+2?"}
"""

In [22]:
prompt2 = f"""
Please answer the user's question:

{"Whats 2+2?"}

Answer the question with ample detail
"""

In [24]:
messages = []

add_user_message(messages, prompt2)
chat(messages, system=None)

'# What is 2 + 2?\n\nThe answer to 2 + 2 is **4**.\n\n## Detailed Explanation\n\n### Basic Arithmetic\nThis is a fundamental addition problem in mathematics. When you combine 2 units with another 2 units, you get a total of 4 units.\n\n### Different Ways to Understand It\n\n1. **Counting**: If you have 2 objects (like apples) and someone gives you 2 more apples, counting them all together gives you 4 apples total.\n\n2. **Number Line**: Starting at 2 on a number line and moving 2 spaces to the right lands you on 4.\n\n3. **Visual Representation**:\n   - ●● + ●● = ●●●●\n   - Two dots plus two dots equals four dots\n\n### Mathematical Properties Demonstrated\n- This problem demonstrates the **commutative property** of addition: 2 + 2 = 2 + 2 (the order doesn\'t matter)\n- It also shows **closure** under addition: adding two whole numbers gives another whole number\n\n### Cultural Significance\nThe equation "2 + 2 = 4" is often used as an example of an objective, undeniable truth in philo

### Generating Test datsets 

First we need our helper functions

In [43]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import json

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [44]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, stop_sequences=None, max_tokens=4000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    if message.stop_reason == "max_tokens":
        print("WARNING: truncated")
    return message.content[0].text

In [45]:
def generate_dataset(n=3):
    prompt = f"""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used
to evaluate prompts that generate Python, JSON, or Regex specifically for
AWS-related tasks. Generate an array of JSON objects, each representing a task
that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {{"task": "Description of task"}}
]
```

* Focus on tasks solvable by a single Python function, JSON object, or regex
* Focus on tasks that do not require writing much code

Please generate {n} objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip())

In [46]:
dataset = generate_dataset(3)

for item in dataset:
    print("-", item["task"])

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"\nSaved {len(dataset)} tasks")

- Create a JSON configuration object for an AWS S3 bucket policy that allows public read access to all objects
- Write a Python function that extracts the AWS region from an S3 bucket ARN using regex
- Create a JSON object representing an AWS Lambda function environment variables configuration with database credentials

Saved 3 tasks


### Running the Eval

In [47]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [48]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [49]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [50]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [51]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Policy for Public Read Access\n\nHere's a JSON configuration object for an S3 bucket policy that allows public read access:\n\n```json\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Sid\": \"PublicReadGetObject\",\n      \"Effect\": \"Allow\",\n      \"Principal\": \"*\",\n      \"Action\": \"s3:GetObject\",\n      \"Resource\": \"arn:aws:s3:::your-bucket-name/*\"\n    }\n  ]\n}\n```\n\n## Key Components Explained\n\n| Component | Description |\n|-----------|-------------|\n| **Version** | Policy language version (use `2012-10-17`) |\n| **Statement** | Array containing policy rules |\n| **Sid** | Statement ID (optional, but recommended) |\n| **Effect** | `Allow` or `Deny` - permits public read access |\n| **Principal** | `\"*\"` means anyone/public access |\n| **Action** | `s3:GetObject` allows reading/downloading objects |\n| **Resource** | `arn:aws:s3:::bucket-name/*` targets all objects in the bucket |\n\n## Important Cons